# **Preprocessing the CERT Dataset**:

### Imports & Constants:

In [1]:
import config
from scripts.Preprocessing import *
from config import CERT_PATH ,DEFAULT_OUTPUT_DIR
MV = config.MODEL_VERSION

In [ ]:
WORK_HOURS = (9, 17)
LONG_URL_THRESHOLD = 90

### Building Layer A: (User, PC, Day) Level Dataset

In [ ]:
layer_a_dataset, nunique_frames = build_layer_a(
    cert_path=CERT_PATH,
    work_hours=WORK_HOURS,
    return_nunique_frames=True,
    compute_schedules=True,
    save_schedule_to=os.path.join("processed_datasets", DEFAULT_OUTPUT_DIR, "user_work_hours.parquet")
)

In [ ]:
layer_a_dataset.head()

In [ ]:
layer_a_path = save_dataset(
    dataset=layer_a_dataset,
    filename=f"ueba_dataset_{MV}a.csv",
    output_dir=DEFAULT_OUTPUT_DIR
)

### Building Layer B: (User, Day) Model Training Dataset

In [ ]:
# Loading the LDAP metadata
ldap_df = load_ldap(CERT_PATH)

In [ ]:
layer_b_dataset = build_layer_b(
    layer_a_df=layer_a_dataset,
    rolling_window=5,
    nunique_frames=nunique_frames,
    ldap_df=ldap_df,
    peer_col=config.PEER_GROUP_KEY,
)

In [ ]:
layer_b_dataset.head()

In [ ]:
layer_b_path = save_dataset(
    dataset=layer_b_dataset,
    filename=f"ueba_dataset_{MV}b.csv",
    output_dir=DEFAULT_OUTPUT_DIR
)

### Creating the Chronological Train/Test Split

In [ ]:
# Splitting dataset B
train_df, test_df = chronological_split(df=layer_b_dataset, split_ratio=0.9)

In [ ]:
# Saving training and testing sets
train_path = save_dataset(train_df, f"ueba_dataset_{MV}_train.csv", DEFAULT_OUTPUT_DIR)
test_path = save_dataset(test_df, f"ueba_dataset_{MV}_test_stream.csv", DEFAULT_OUTPUT_DIR)

In [ ]:
# Parquet copies for downstream notebooks (Autoencoder reads parquet)
train_df.to_parquet(train_path.replace(".csv", ".parquet"), index=False)
test_df.to_parquet(test_path.replace(".csv", ".parquet"), index=False)
print(f"Parquet versions saved alongside CSVs.")

###  Utilizing Drill-Down Dataset Lookup

In [ ]:
# Example usage
user = "abh0349"
day = "2010-01-27"

In [ ]:
drill_down_table = get_drilldown(layer_a_dataset, user, day)

In [ ]:
drill_down_table